In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

In [ ]:
%load_ext autoreload
%autoreload 2

import sys; sys.path.append('..')
import MeshFEM, mesh, elastic_sheet, elastic_solid, energy, tensors, benchmark
import meshing, triangulation, py_newton_optimizer
import tri_mesh_viewer
from io_redirection import suppress_stdout as so
import sheet_convergence, sim_utils, semisphere_convergence
import numpy as np, time
import copy

In [ ]:
thickness = 0.01
mV = 1e-5
mA = 1e-5
myL = 1.2
myH = 0.5
s = 1 - 0.01

In [ ]:
m = semisphere_convergence.getRecSheetMesh(L = myL, H = myH, maxArea=mA)
esheet = semisphere_convergence.getElasticSheet(m,thickness,useCreases=False)
esheet.hessianProjectionType = esheet.hessianProjectionType.MembraneFBased

In [ ]:
sheet_viewer = tri_mesh_viewer.Viewer(esheet, wireframe=True)
sheet_viewer.show()

In [ ]:
# Apply the strain condition
esheet.setDeformedPositions(esheet.getRestPositions() @ np.diag([s, 1, 1]))

In [ ]:
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.niter = 100
opts.gradTol = 1e-10

In [ ]:
def getFixedVars(obj):
    """
    Pin down rigid motion by fixing displaccement components of the four midsurface corners.
    """
    tol = 1e-8
    X = obj.getRestPositions()
    xmin_ymin_vtx = np.where((np.abs(X[:, 0] - X[:, 0].min()) < tol) * (np.abs(X[:, 1] - X[:, 1].min()) < tol))[0][0]
    xmax_ymin_vtx = np.where((np.abs(X[:, 0] - X[:, 0].max()) < tol) * (np.abs(X[:, 1] - X[:, 1].min()) < tol))[0][0]
    xmin_ymax_vtx = np.where((np.abs(X[:, 0] - X[:, 0].min()) < tol) * (np.abs(X[:, 1] - X[:, 1].max()) < tol))[0][0]
    xmax_ymax_vtx = np.where((np.abs(X[:, 0] - X[:, 0].max()) < tol) * (np.abs(X[:, 1] - X[:, 1].max()) < tol))[0][0]
    fixedVars  = [3 * xmin_ymin_vtx + 2, 3 * xmin_ymax_vtx + 2, 3 * xmax_ymin_vtx + 2, 3 * xmax_ymax_vtx + 2] # Keep left and right edges on the "ground" (also preventing rotations around x/y axes)
    fixedVars += [3 * xmin_ymin_vtx + 1, 3 * xmax_ymin_vtx + 1] # Pin global y translation and rotation around z axis
    return fixedVars

def getLoads(obj):
    """
    Use springs pulling the entire left/right faces toward the target compressed/stretched positions.
    These are preferable to Dirichlet constraints as they apply a constant traction boundary
    condition, avoiding the local stress oscillations caused by Dirichlet constraints in the
    `ElasticSheet` simulation and stress concentrations in the `ElasticSolid` simulation
    (where Dirichlet constraints at the midsurface act like "needles").
    """
    import loads
    apc_xmin = sim_utils.getBoundaryFaceCentroidAttachmentPointCoordinate(obj, sim_utils.BBoxFace.MIN_X)
    apc_xmax = sim_utils.getBoundaryFaceCentroidAttachmentPointCoordinate(obj, sim_utils.BBoxFace.MAX_X)
    return [loads.Springs(obj, [apc_xmin, apc_xmax], [loads.AttachmentPointCoordinate(0), loads.AttachmentPointCoordinate(s * myL)], [200000, 200000])]

In [ ]:
# Break symmetry for faster convergence
esheet.setVars(esheet.getVars() + 1e-6 * np.random.normal(size=esheet.numVars()))

In [ ]:
benchmark.reset()
esheet.computeEquilibrium(loads=getLoads(esheet), fixedVars=getFixedVars(esheet), opts=opts)#, cb=lambda p, it: sheet_viewer.update())
benchmark.report()
sheet_viewer.update()

In [ ]:
rec_halfmesh = semisphere_convergence.getRecTetMesh(thickness/2, L=myL, H=myH,maxVol=0.1)
rec_halfmesh.setVertices(rec_halfmesh.vertices() + [0, 0, thickness / 4]) # Shift top half up to cover the z interval [0, thickness / 2]

import mesh_operations
reflect_result = mesh_operations.reflectMesh(rec_halfmesh.vertices(),rec_halfmesh.elements(),axes=[2])

reflectedmesh = mesh.Mesh(*reflect_result, degree=2)

esolid = semisphere_convergence.getElasticSolid(reflectedmesh, useNeoHookean=True)

In [ ]:
solid_viewer = tri_mesh_viewer.Viewer(esolid, wireframe=True)
solid_viewer.show()

In [ ]:
# Break symmetry in a controlled way (set the sign based on whether the sheet simulation popped "up" or "down")
x = esolid.getDeformedPositions()
x[:, 2] += -1e-6 * (0.5**2 -(esolid.getRestPositions()[:, 0] / myL - 0.5)**2)
esolid.setDeformedPositions(x)

In [ ]:
benchmark.reset()
cr = esolid.computeEquilibrium(loads=getLoads(esolid), fixedVars=getFixedVars(esolid), opts=opts, cb=lambda p, it: solid_viewer.update())
benchmark.report()

In [ ]:
esolid_energy = esolid.energy()
esheet_energy = esheet.energy()
energy_rel_error = np.abs(esolid_energy-esheet_energy)/esolid_energy
print("Relative Error of Energy: ",energy_rel_error*100, "%")

In [ ]:
# Visualization

In [ ]:
from ipywidgets import HBox
sheetView, sampledSolidView, vdefo = semisphere_convergence.visVertexStressOnSheetAndSolid(esolid, esheet, scalarMeasure=semisphere_convergence.minEigenvalue, h=0)
HBox([sheetView.show(), sampledSolidView.show(), vdefo.show()])

In [ ]:
vertex_stress_rel_error,_,_,_,_,_ = semisphere_convergence.computeVertexStresses(esolid, esheet, h=thickness/2, scalarMeasure=semisphere_convergence.maxEigenvalue)
print("Relative Error of Vertex Stress: ", vertex_stress_rel_error*100, "%")

In [ ]:
vertex_stress_rel_error,_,_,_,_,_ = semisphere_convergence.computeVertexStresses(esolid, esheet, h=thickness/2, scalarMeasure=semisphere_convergence.fullTensor)
print("Relative Error of Vertex Stress: ", vertex_stress_rel_error*100, "%")

In [ ]:
vertex_stress_rel_error,_,_,_,_,_ = semisphere_convergence.computeVertexStresses(esolid, esheet, h=-thickness/2, scalarMeasure=semisphere_convergence.fullTensor)
print("Relative Error of Vertex Stress: ", vertex_stress_rel_error*100, "%")

In [ ]:
vertex_stress_rel_error,_,_,_,_,_ = semisphere_convergence.computeVertexStresses(esolid, esheet, h=0, scalarMeasure=semisphere_convergence.fullTensor)
print("Relative Error of Vertex Stress: ", vertex_stress_rel_error*100, "%")

In [ ]:
vertex_stress_rel_error,_,_,_,_,_ = semisphere_convergence.computeVertexStresses(esolid, esheet, h=0, scalarMeasure=semisphere_convergence.frobeniusNorm)
print("Relative Error of Vertex Stress: ", vertex_stress_rel_error*100, "%")

In [ ]:
vertex_stress_rel_error,_,_,_,_,_ = semisphere_convergence.computeVertexStresses(esolid, esheet, h=0, scalarMeasure=semisphere_convergence.maxEigenvalue)
print("Relative Error of Vertex Stress: ", vertex_stress_rel_error*100, "%")

In [ ]:
vertex_stress_rel_error,_,_,_,_,_ = semisphere_convergence.computeVertexStresses(esolid, esheet, h=0, scalarMeasure=semisphere_convergence.minEigenvalue)
print("Relative Error of Vertex Stress: ", vertex_stress_rel_error*100, "%")

In [ ]:
# Write stress fields to a `.msh` file that can be viewed in Gmsh
vertex_stress_rel_error,sheetStress,_,sampledSolidStress,_,_ = semisphere_convergence.computeVertexStresses(esolid, esheet, h=0, scalarMeasure=semisphere_convergence.fullTensor)
mfw = mesh.MSHFieldWriter("compare.msh", esheet.getDeformedPositions(), esheet.mesh().elements())
mfw.addField('sheetStress', sheetStress.reshape(-1, 9))
mfw.addField('sampledSolidStress', sampledSolidStress.reshape(-1, 9))
del mfw